# 段階的検証：上位解法を1つずつ追加して提出ファイル作成

**目的**
- ベースライン（train_baseline）に、上位解法を**段階的に**追加する
- 各段階ごとに提出用CSVを1つずつ作り、LBで「どの追加が効くか」を検証する
- **既知/未知でモデルを分ける**パターンを中心に試す

**流れ**
1. セットアップ・データ読み込み → 全特徴量を1回追加（`add_all_features`）
2. 下のステージを順に実行（または「全ステージ一括」のセルを実行）
3. `outputs/submissions/staged/` に各段階の提出が保存されるので、それぞれ提出してLBで比較

**既知/未知** = train に登場した映画かどうかで「既知用モデル」と「未知用モデル」を切り替えて予測。ステージ 6・7・8 で特徴量の強さを変えながら試せる。

**ステージ一覧**
| ステージ | 内容 | モデル |
|----------|------|--------|
| 0 | ベースライン38のみ | 1本 |
| 1〜5 | v2 → v2_ratio → v2_nmf → v2_svd → v2_all | 各1本 |
| 6 | **既知/未知（ベースライン38）** | 2本（既知用・未知用） |
| 7 | **既知/未知（v2）** | 2本 |
| 8 | **既知/未知（v2_all）** | 2本 |
| 9 | v2_all シード3本平均 | 3本の平均 |

ステージ 1〜5 で追加する特徴量の詳細は `docs/PROJECT_MIND.md` および `archive/top_solutions.py` を参照。

In [ ]:
# archive/ から実行しても lib と top_solutions を参照できるようにする（カーネルはプロジェクトルートで起動するか、このセルを最初に実行）
import sys
from pathlib import Path
_root = Path.cwd().parent if not (Path.cwd() / "lib").exists() else Path.cwd()
_archive = Path.cwd() if _root != Path.cwd() else Path.cwd() / "archive"
for p in (str(_root), str(_archive)):
    if p not in sys.path:
        sys.path.insert(0, p)

In [1]:
import os
import random
import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore")

from lib import get_baseline_data, BASELINE_FEATURES, BASELINE_LGB_PARAMS
from top_solutions import (
    add_all_features,
    save_submission,
    get_unknown_features_simple,
    SEED_AVERAGING_SEEDS,
)

OUTPUT_DIR = "outputs"
STAGED_DIR = os.path.join(OUTPUT_DIR, "submissions", "staged")
os.makedirs(STAGED_DIR, exist_ok=True)
print("Setup complete.")

Setup complete.


In [2]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)

seed_everything(42)

## データ読み込み・特徴量追加

In [3]:
train, test = get_baseline_data()
y = train["target"].values
baseline_features = list(BASELINE_FEATURES)

_, _, feature_sets = add_all_features(train, test, y, baseline_features)
params = BASELINE_LGB_PARAMS

print(f"Train: {len(train):,}, Test: {len(test):,}")
print("feature_sets:", list(feature_sets.keys()))

Train: 653,507, Test: 40,716
feature_sets: ['baseline', 'v2', 'v2_ratio', 'v2_nmf', 'v2_svd', 'v2_all', 'known', 'unknown']


## ステージ 0: ベースライン38のみ（1本）

In [4]:
feats = feature_sets["baseline"]
model = lgb.LGBMClassifier(**params)
model.fit(train[feats], y)
pred = model.predict_proba(test[feats])[:, 1]
path = os.path.join(STAGED_DIR, "submission_stage0_baseline38.csv")
save_submission(test, pred, path)
print(f"Saved {path}")

Saved outputs/submissions/staged/submission_stage0_baseline38.csv


## ステージ 1〜5: 特徴量を段階的に追加（各1本）

In [5]:
stages_1_5 = [
    ("v2", "submission_stage1_v2.csv"),
    ("v2_ratio", "submission_stage2_v2_ratio.csv"),
    ("v2_nmf", "submission_stage3_v2_nmf.csv"),
    ("v2_svd", "submission_stage4_v2_svd.csv"),
    ("v2_all", "submission_stage5_v2_all.csv"),
]
for config_key, filename in stages_1_5:
    feats = feature_sets[config_key]
    model = lgb.LGBMClassifier(**params)
    model.fit(train[feats], y)
    pred = model.predict_proba(test[feats])[:, 1]
    path = os.path.join(STAGED_DIR, filename)
    save_submission(test, pred, path)
    print(f"Saved {path}")
print("Stage 1〜5 done.")

Saved outputs/submissions/staged/submission_stage1_v2.csv
Saved outputs/submissions/staged/submission_stage2_v2_ratio.csv
Saved outputs/submissions/staged/submission_stage3_v2_nmf.csv
Saved outputs/submissions/staged/submission_stage4_v2_svd.csv
Saved outputs/submissions/staged/submission_stage5_v2_all.csv
Stage 1〜5 done.


## ステージ 6: 既知/未知（ベースライン38）

既知映画用＝38特徴、未知映画用＝38のうち映画IDを除いた特徴でそれぞれ1本ずつ学習し、testで振り分け。

In [6]:
feats_known = feature_sets["baseline"]
feats_unknown = get_unknown_features_simple(feats_known)

train_movies = set(train["rotten_tomatoes_link"])
test["_is_known"] = test["rotten_tomatoes_link"].isin(train_movies)

model_k = lgb.LGBMClassifier(**params)
model_k.fit(train[feats_known], y)
model_u = lgb.LGBMClassifier(**params)
model_u.fit(train[feats_unknown], y)

pred = np.where(
    test["_is_known"].values,
    model_k.predict_proba(test[feats_known])[:, 1],
    model_u.predict_proba(test[feats_unknown])[:, 1],
)
test.drop(columns=["_is_known"], inplace=True)

path = os.path.join(STAGED_DIR, "submission_stage6_known_unknown_baseline38.csv")
save_submission(test, pred, path)
print(f"Saved {path}")

Saved outputs/submissions/staged/submission_stage6_known_unknown_baseline38.csv


## ステージ 7: 既知/未知（v2）

既知用＝v2特徴、未知用＝v2から映画ID・映画TE・レビュー数を除いた特徴で2本。

In [7]:
feats_known = feature_sets["v2"]
feats_unknown = get_unknown_features_simple(feats_known)

train_movies = set(train["rotten_tomatoes_link"])
test["_is_known"] = test["rotten_tomatoes_link"].isin(train_movies)

model_k = lgb.LGBMClassifier(**params)
model_k.fit(train[feats_known], y)
model_u = lgb.LGBMClassifier(**params)
model_u.fit(train[feats_unknown], y)

pred = np.where(
    test["_is_known"].values,
    model_k.predict_proba(test[feats_known])[:, 1],
    model_u.predict_proba(test[feats_unknown])[:, 1],
)
test.drop(columns=["_is_known"], inplace=True)

path = os.path.join(STAGED_DIR, "submission_stage7_known_unknown_v2.csv")
save_submission(test, pred, path)
print(f"Saved {path}")

Saved outputs/submissions/staged/submission_stage7_known_unknown_v2.csv


## ステージ 8: 既知/未知（v2_all）

既知用＝v2_all、未知用＝映画ID等を除きNMF/SVDの映画側埋め込みは使う（top_solutions の known_unknown と同じ）。

In [8]:
feats_known = feature_sets["known"]
feats_unknown = feature_sets["unknown"]

train_movies = set(train["rotten_tomatoes_link"])
test["_is_known"] = test["rotten_tomatoes_link"].isin(train_movies)

model_k = lgb.LGBMClassifier(**params)
model_k.fit(train[feats_known], y)
model_u = lgb.LGBMClassifier(**params)
model_u.fit(train[feats_unknown], y)

pred = np.where(
    test["_is_known"].values,
    model_k.predict_proba(test[feats_known])[:, 1],
    model_u.predict_proba(test[feats_unknown])[:, 1],
)
test.drop(columns=["_is_known"], inplace=True)

path = os.path.join(STAGED_DIR, "submission_stage8_known_unknown_v2_all.csv")
save_submission(test, pred, path)
print(f"Saved {path}")

Saved outputs/submissions/staged/submission_stage8_known_unknown_v2_all.csv


## ステージ 9: v2_all シード3本平均

In [9]:
feats = feature_sets["v2_all"]
all_preds = []
for seed in SEED_AVERAGING_SEEDS:
    model = lgb.LGBMClassifier(**{**params, "random_state": seed})
    model.fit(train[feats], y)
    all_preds.append(model.predict_proba(test[feats])[:, 1])
pred = np.mean(all_preds, axis=0)

path = os.path.join(STAGED_DIR, "submission_stage9_v2_all_seed3.csv")
save_submission(test, pred, path)
print(f"Saved {path}")

Saved outputs/submissions/staged/submission_stage9_v2_all_seed3.csv


## （オプション）全ステージを一括実行

In [ ]:
# 上から順に実行する代わりに、このセル1つで全提出ファイルを作成
def run_all_stages():
    params = BASELINE_LGB_PARAMS
    # 0
    f = feature_sets["baseline"]
    m = lgb.LGBMClassifier(**params)
    m.fit(train[f], y)
    save_submission(test, m.predict_proba(test[f])[:, 1], os.path.join(STAGED_DIR, "submission_stage0_baseline38.csv"))
    # 1-5
    for config, name in [("v2", "stage1_v2"), ("v2_ratio", "stage2_v2_ratio"), ("v2_nmf", "stage3_v2_nmf"), ("v2_svd", "stage4_v2_svd"), ("v2_all", "stage5_v2_all")]:
        f = feature_sets[config]
        m = lgb.LGBMClassifier(**params)
        m.fit(train[f], y)
        save_submission(test, m.predict_proba(test[f])[:, 1], os.path.join(STAGED_DIR, f"submission_{name}.csv"))
    # 6: known_unknown baseline
    fk, fu = feature_sets["baseline"], get_unknown_features_simple(feature_sets["baseline"])
    test["_ik"] = test["rotten_tomatoes_link"].isin(set(train["rotten_tomatoes_link"]))
    mk = lgb.LGBMClassifier(**params); mk.fit(train[fk], y)
    mu = lgb.LGBMClassifier(**params); mu.fit(train[fu], y)
    save_submission(test, np.where(test["_ik"].values, mk.predict_proba(test[fk])[:, 1], mu.predict_proba(test[fu])[:, 1]), os.path.join(STAGED_DIR, "submission_stage6_known_unknown_baseline38.csv"))
    test.drop(columns=["_ik"], inplace=True)
    # 7: known_unknown v2
    fk, fu = feature_sets["v2"], get_unknown_features_simple(feature_sets["v2"])
    test["_ik"] = test["rotten_tomatoes_link"].isin(set(train["rotten_tomatoes_link"]))
    mk = lgb.LGBMClassifier(**params); mk.fit(train[fk], y); mu = lgb.LGBMClassifier(**params); mu.fit(train[fu], y)
    save_submission(test, np.where(test["_ik"].values, mk.predict_proba(test[fk])[:, 1], mu.predict_proba(test[fu])[:, 1]), os.path.join(STAGED_DIR, "submission_stage7_known_unknown_v2.csv"))
    test.drop(columns=["_ik"], inplace=True)
    # 8: known_unknown v2_all
    fk, fu = feature_sets["known"], feature_sets["unknown"]
    test["_ik"] = test["rotten_tomatoes_link"].isin(set(train["rotten_tomatoes_link"]))
    mk = lgb.LGBMClassifier(**params); mk.fit(train[fk], y); mu = lgb.LGBMClassifier(**params); mu.fit(train[fu], y)
    save_submission(test, np.where(test["_ik"].values, mk.predict_proba(test[fk])[:, 1], mu.predict_proba(test[fu])[:, 1]), os.path.join(STAGED_DIR, "submission_stage8_known_unknown_v2_all.csv"))
    test.drop(columns=["_ik"], inplace=True)
    # 9: v2_all seed3
    f = feature_sets["v2_all"]
    preds = [lgb.LGBMClassifier(**{**params, "random_state": s}).fit(train[f], y).predict_proba(test[f])[:, 1] for s in SEED_AVERAGING_SEEDS]
    save_submission(test, np.mean(preds, axis=0), os.path.join(STAGED_DIR, "submission_stage9_v2_all_seed3.csv"))
run_all_stages()
print("全10ファイル作成完了.")

## 作成された提出ファイル一覧

In [ ]:
staged_files = [
    "submission_stage0_baseline38.csv",
    "submission_stage1_v2.csv",
    "submission_stage2_v2_ratio.csv",
    "submission_stage3_v2_nmf.csv",
    "submission_stage4_v2_svd.csv",
    "submission_stage5_v2_all.csv",
    "submission_stage6_known_unknown_baseline38.csv",
    "submission_stage7_known_unknown_v2.csv",
    "submission_stage8_known_unknown_v2_all.csv",
    "submission_stage9_v2_all_seed3.csv",
]
for f in staged_files:
    p = os.path.join(STAGED_DIR, f)
    exists = "OK" if os.path.exists(p) else "---"
    print(f"  {exists} {f}")
print(f"\n→ {STAGED_DIR}")